In [7]:
import numpy as np
import pandas as pd
import math
import requests
import geopy.distance

In [14]:
# Set the search parameters
population_min = 5000
population_max = 50000
costco_max_distance = 30
airport_max_distance = 30 #miles
population_density_max = 1500 # people per sq mile


# Load the list of cities and their coordinates
cities_df = pd.read_csv('uscities.csv', usecols=['city','state_id','county_fips', 'lat', 'lng', 'population','density','zips'])

## Filter cities by population
cities_df = cities_df[(cities_df['population'] >= population_min) & (cities_df['population'] <= population_max)]

# Filter cities by population density
cities_df = cities_df[(cities_df['density'] <= population_density_max)]

# # Filter cities by state ID
# states = ['CA', 'NY']
# cities_df = cities_df[(cities_df['state_id'].isin(states))]

# Load the list of airports and their coordinates
airports_df = pd.read_csv('airports.csv', usecols=['type','name','latitude_deg', 'longitude_deg', 'iso_country', 'iso_region'])
airports_df = airports_df[(airports_df['type'] == "large_airport") & (airports_df['iso_country'] == "US")]
# airports_df = airports_df[((airports_df['type'] == "large_airport") | (airports_df['type'] == "medium_airport")) & (airports_df['iso_country'] == "US")]



In [15]:
# function to calculate distance between two points on Earth's surface using Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # radius of the Earth in kilometers
    dLat = math.radians(lat2 - lat1)
    dLon = math.radians(lon2 - lon1)
    lat1 = math.radians(lat1)
    lat2 = math.radians(lat2)
    a = math.sin(dLat/2)**2 + math.sin(dLon/2)**2 * math.cos(lat1) * math.cos(lat2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    d = R * c
    return d

## Calculate distances between cities and airports
# initialize distances array
distances = np.zeros((len(cities_df), len(airports_df)))

# calculate distances between cities and airports
for i in range(len(cities_df)):
    for j in range(len(airports_df)):
        d = haversine(cities_df.iloc[i]['lat'], cities_df.iloc[i]['lng'], airports_df.iloc[j]['latitude_deg'], airports_df.iloc[j]['longitude_deg'])
        # Convert to miles
        d = d * 0.621371
        distances[i, j] = d

# Find the closest airport for each city
city_airport = []
city_airport_distance = []
for i in range(len(distances)):
    closest_airport_index = np.argmin(distances[i])
    closest_airport_name = airports_df.iloc[closest_airport_index]['name']
    closest_airport_distance = distances[i][closest_airport_index]
    city_airport.append(closest_airport_name)
    city_airport_distance.append(closest_airport_distance)

# Append the closest airport and its distance to cities_df
cities_df['closest_airport'] = city_airport
cities_df['airport_distance'] = city_airport_distance

## Filter cities by airport distance
cities_df = cities_df[(cities_df['airport_distance'] <= airport_max_distance)]


In [16]:
# save cities_df to Excel
cities_df.to_excel('cities.xlsx')
print(len(cities_df)) #number of cities

1607
